In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam

w = WorkspaceClient()

# 1. Create/identify the group that acts as the "role"
role_group = w.groups.create(display_name="finance-role")
print(f"Created group: {role_group.display_name}, ID: {role_group.id}")

# 2. Build the resource name for the rule set
# Format: accounts/{account_id}/groups/{group_id}/ruleSets/default
account_id = w.config.account_id  # or hardcode your account_id string
rule_set_name = f"accounts/{account_id}/groups/{role_group.id}/ruleSets/default"

# 3. Get the current rule set first — REQUIRED to obtain the etag
current_rule_set = w.access_control.get_rule_set(
    name=rule_set_name,
    etag="",  # empty string fetches the latest version
)
print(f"Current etag: {current_rule_set.etag}")

# 4. Define who can Assume this role (group acting as a role)
principal_to_grant = "55497374-cd30-45b5-8476-eb881e25214d"       # a user
# principal_to_grant = "groups/other-group-id"               # or another group
# principal_to_grant = "servicePrincipals/<application_id>"  # or a service principal

grant_rule = iam.GrantRule(
    role="roles/group.assume",     # role that grants "Assume" permission on the group
    principals=["55497374-cd30-45b5-8476-eb881e25214d"],
)

# 5. Update the rule set with the new grant rule
updated_rule_set = w.access_control.update_rule_set(
    name=rule_set_name,
    rule_set=iam.RuleSetUpdateRequest(
        name=rule_set_name,
        etag=current_rule_set.etag,   # required — prevents overwriting concurrent changes
        grant_rules=[grant_rule],
    ),
)

print("Rule set updated successfully")
print(updated_rule_set)